In [ ]:
import copy
import math
import os
import re
import sys
import time
from collections import defaultdict
from glob import glob

import numpy as np
import pandas as pd
import xarray as xr
import yaml
import zarr

ds_static = xr.open_zarr('/glade/derecho/scratch/ksha/EPRI_data/static/static.zarr')
lon = ds_static['lon'].values
lat = ds_static['lat'].values

dict_loc = {
    'Pituffik': (76.4, -68.575),
    'Fairbanks': (64.75, -147.4),
    'Guam': (13.475, 144.75),
    'Yuma_PG': (33.125, -114.125),
    'Fort_Bragg': (35.05, -79.115),
}

def time_to_lead_and_stack(ds_list, new_dim="member", labels=None, ref="first"):
    """
    ds_list: list[xr.Dataset], each has a 'time' coord with daily values
    new_dim: name of the new dimension to stack on
    labels: optional labels for new_dim (len == len(ds_list))
    ref: "first" (per-dataset first time) or a numpy/pandas datetime-like scalar
    """
    out = []
    for i, ds in enumerate(ds_list):
        ds = ds.sortby("time")

        if ref == "first":
            t0 = ds["time"].isel(time=0)
        else:
            # global reference (same for all ds)
            t0 = xr.DataArray(ref)

        lead = (ds["time"] - t0).astype("timedelta64[D]")  # daily deltas
        ds2 = ds.assign_coords(lead_time=("time", lead.data)).swap_dims({"time": "lead_time"})
        ds2 = ds2.drop_vars("time")  # optional: remove original coordinate

        out.append(ds2)

    if labels is None:
        labels = list(range(len(out)))

    stacked = xr.concat(out, dim=xr.IndexVariable(new_dim, labels))
    return stacked


# ============================================================================= #
# Gather CESM-SMYLE daily data
# ============================================================================= #

years = np.arange(1958, 2020, 1)
varnames = ['FLDS', 'FSDS', 'PRECSC', 'PRECSL', 'PRECT', 'PSL', 'QREFHT', 'TMQ', 'TREFHT', 'TREFHTMN', 'TREFHTMX', 'U10']


for year_init in years:

    year_start = year_init + 1
    year_end = year_init + 10

    fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM2_SMYLE/SMYLE_{year_init}-11-01_daily_ensemble.zarr'
    ds = xr.open_zarr(fn)[varnames]
    ds = ds.sel(time=slice(f'{year_start}-01-01T00', f'{year_end}-12-31T00'))

    ds = ds.assign_coords(lon = (((ds.lon + 180) % 360) - 180))
    ds = ds.sortby("lon")
    
    for stn, (lat_mid, lon_mid) in dict_loc.items():
        
        subset = ds.sel(lat=lat_mid, lon=lon_mid, method='nearest')
        subset = subset.mean(('member',))
        
        L = len(subset['time'])
        subset = subset.chunk({'time': L})
        
        save_name = f'/glade/campaign/ral/hap/ksha/EPRI_data/CESM_SMYLE_STN/{stn}_{year_init}.zarr'        
        subset.to_zarr(save_name, mode='w', consolidated=True, compute=True)
        print(save_name)

# ============================================================================= #
# Convert CESM-SMYLE daily to annual metrics
# ============================================================================= #

station_names = ['Pituffik', 'Fairbanks', 'Guam', 'Yuma_PG' ,'Fort_Bragg'] # 

for stn in station_names:
    
    t0 = time.perf_counter()
    
    base_dir = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS_STN/{stn}/'
    
    # ========================== #
    # get data
    list_ds = []
    for year in range(1958, 2020):
        fn = f'/glade/campaign/ral/hap/ksha/EPRI_data/CESM_SMYLE_STN/{stn}_{year}.zarr'
        ds = xr.open_zarr(fn)[['PRECT', 'TREFHT', 'TREFHTMX', 'TREFHTMN']]
        list_ds.append(ds)
        
    ds_all = time_to_lead_and_stack(list_ds, new_dim="init_time")
    
    ds_all['PRECT'] = ds_all['PRECT'] * 60*60*24 * 1000 # mm per day
    
    ds_all = ds_all.assign_coords({'init_time': np.arange(1959, 2021)})
    lead_year = (ds_all["lead_time"] / np.timedelta64(365, "D")).astype(int)
    ds_all = ds_all.assign_coords(lead_year=("lead_time", lead_year.data))
    
    # ======================= #
    # metrics
    ds_group = ds_all.groupby("lead_year")
    ds_max  = ds_group.max(dim="lead_time",  skipna=True)
    ds_min  = ds_group.min(dim="lead_time",  skipna=True)
    ds_mean  = ds_group.mean(dim="lead_time",  skipna=True)
    ds_30d = ds_group.map(
        lambda x: x.rolling(lead_time=30, min_periods=30).mean().max(dim="lead_time", skipna=True)
    )
    ds_min = ds_min.rename({'TREFHTMN': 'TREFHTMN_min', 'TREFHT': 'TREFHT_min'})[['TREFHTMN_min', 'TREFHT_min']]
    ds_max = ds_max.rename({'PRECT': 'PRECT_max', 'TREFHTMX': 'TREFHTMX_max', 'TREFHT': 'TREFHT_max'})[['PRECT_max', 'TREFHTMX_max', 'TREFHT_max']]
    ds_30d = ds_30d.rename({'TREFHT': 'TREFHT_30d', 'PRECT': 'PRECT_30d'})[['TREFHT_30d', 'PRECT_30d']]
    ds_mean = ds_mean.rename({'PRECT': 'PRECT_mean', 'TREFHT': 'TREFHT_mean'})[['PRECT_mean', 'TREFHT_mean']]
    ds_metrics = xr.merge([ds_min, ds_max, ds_30d, ds_mean])
    ds_metrics = ds_metrics.rename({v: f"{v}_default" for v in ds_metrics.data_vars})
    
    # ========================== #
    # save
    ds_final = ds_metrics
    #xr.merge([ds_metrics, ds_metrics_anom, ds_metrics_detrend])
    save_name = base_dir + 'CESM_metrics.zarr'
    ds_final.to_zarr(save_name, mode='w')
    print(save_name)
    
    t1 = time.perf_counter()
    print(f"Elapsed: {t1 - t0:.6f} s")


# ============================================================================= #
# Compute CESM Greenland Blocking Index (GBI)
# ============================================================================= #

list_GBI = []

ilead = 0
# for ilead in range(10):
list_ds = []

for year in range(1958, 2020):
    # get data
    fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM2_SMYLE/SMYLE_{year}-11-01_daily_ensemble.zarr'
    ds = xr.open_zarr(fn)[['Z500']]
    ds = ds.mean('member')
    ds = ds.sel(time=slice(f'{year+1}-01-01', f'{year+10}-12-31'))

    # GBI definition
    Z500 = ds['Z500']
    Z500 = Z500.assign_coords(lon=((Z500.lon + 180) % 360) - 180).sortby("lon")
    
    # robust lat slicing
    if Z500.lat[0] < Z500.lat[-1]:
        Z500_GBI = Z500.sel(lat=slice(60, 80), lon=slice(-80, -20))
    else:
        Z500_GBI = Z500.sel(lat=slice(80, 60), lon=slice(-80, -20))
    
    Z500_mean = Z500.sel(lat=slice(60, 80))
    
    GBI = Z500_GBI.mean(('lat', 'lon')) - Z500_mean.mean(('lat', 'lon'))

    gp = GBI.groupby("time.year")
    GBI_mean  = gp.mean(dim="time",  skipna=True)
    GBI_30d = gp.map(lambda x: x.rolling(time=30, min_periods=30).mean().max(dim="time", skipna=True))

    ds_GBI = xr.Dataset(
        {"GBI_mean": GBI_mean, "GBI_30d": GBI_30d}
    )

    ds_GBI = ds_GBI.rename({'year': 'lead_year'})
    ds_GBI = ds_GBI.assign_coords({'lead_year': np.arange(10)})
    list_GBI.append(ds_GBI)

ds_GBI_all = xr.concat(list_GBI, dim='init_year')
ds_GBI_all = ds_GBI_all.assign_coords({'init_year': np.arange(1958, 2020)})
save_name = '/glade/derecho/scratch/ksha/EPRI_data/METRICS/GBI.zarr'

ds_GBI_all = ds_GBI_all.chunk({'init_year': 62, 'lead_year': 10})
ds_GBI_all.to_zarr(save_name)

# ============================================================================= #
# Compute nino34 from CESM SST
# ============================================================================= #

data_dir = "/glade/derecho/scratch/ksha/EPRI_data/CESM2_SMYLE_OCN/"  # change this
paths = sorted(glob(f"{data_dir}/SMYLE_????-11-01_daily_ensemble.zarr"))

def process_forecast(ds):
    # # --- parse initialization year from filename, e.g. SST_1959.zarr ---
    # m = re.search(r"SMYLE_(\d{4})\-11-01_daily_ensemble.zarr", path)
    
    # init_year = int(m.group(1)) if m else None

    # # --- open Zarr ---
    # ds = xr.open_zarr(path)  # tweak chunking if needed
    # # ds: dims (time: 2981, nlat: 384, nlon: 320)
    # # coords: time, TLAT(nlat,nlon), TLONG(nlat,nlon)
    sst = ds["SST"]

    # ------------------------------------------------------------------
    # (1) SST monthly climatology (for this 10‑year forecast only)
    # ------------------------------------------------------------------
    # Uses all time steps in this forecast, grouped by calendar month
    sst_clim = sst.groupby("time.month").mean("time")    # (month, nlat, nlon)

    # ------------------------------------------------------------------
    # (2) SST anomalies (daily) = SST - monthly climatology
    # ------------------------------------------------------------------
    sst_anom = sst.groupby("time.month") - sst_clim      # (time, nlat, nlon)

    # ------------------------------------------------------------------
    # (3) DJF Niño3.4 index from SST anomalies
    # ------------------------------------------------------------------
    lat = ds["ULAT"]           # (nlat, nlon)
    lon = ds["ULONG"]          # (nlat, nlon)

    # Decide whether longitude is 0–360 or -180–180
    if float(lon.max()) > 180:
        # assume 0–360
        lon_n34 = lon
        lon_min, lon_max = 190, 240    # 170W–120W => 190E–240E
    else:
        # assume -180–180
        lon_n34 = lon
        lon_min, lon_max = -170, -120  # 170W–120W

    # Niño3.4 box: 5S–5N, 170W–120W
    n34_mask = (
        (lat >= -5) & (lat <= 5) &
        (lon_n34 >= lon_min) & (lon_n34 <= lon_max)
    )

    # cos(lat) area weights within Niño3.4
    weights = np.cos(np.deg2rad(lat))
    weights = weights.where(n34_mask)
    
    weights = weights.fillna(0)
    
    # area‑weighted Niño3.4 SST anomaly time series (daily)
    # dims: time
    sst_n34 = (
        sst_anom.where(n34_mask)
                .weighted(weights)
                .mean(dim=("nlat", "nlon"))
    )

    # Convert to monthly mean Niño3.4 anomalies
    n34_mon = sst_n34.resample(time="MS").mean()

    # ---- DJF mean per year ----
    month = n34_mon["time"].dt.month
    year  = n34_mon["time"].dt.year

    # Define "DJF year": Dec of year N belongs to DJF of year N+1
    season_year = xr.where(month == 12, year + 1, year)

    # Keep only Dec–Jan–Feb and average over each DJF season
    n34_DJF = (
        n34_mon
        .where(month.isin([12, 1, 2]))
        .groupby(season_year)
        .mean("time")
    )
    # Rename dim from the default "group" to "year"
    n34_DJF = n34_DJF.rename({"group": "year"})
    
    return n34_DJF

results = {}

for fn in paths:
    ds = xr.open_zarr(fn)
    ds = ds.mean(('member',))

    m = re.search(r"SMYLE_(\d{4})\-11-01_daily_ensemble.zarr", fn)
    init_year = int(m.group(1))
    
    n34_DJF = process_forecast(ds)
    results[init_year] = n34_DJF.values

save_name = '/glade/campaign/ral/hap/ksha/EPRI_data/CESM_OCN/ENSO_index.npy'
np.save(save_name, results)

# ============================================================================= #
# Gather hourly ERA5
# ============================================================================= #

varnames = [
    '2m_temperature',
    '2m_dewpoint_temperature',
    '10m_u_component_of_wind',
    '10m_v_component_of_wind',
    'total_precipitation',
    'minimum_2m_temperature_since_previous_post_processing', 
    'maximum_2m_temperature_since_previous_post_processing', 
    'surface_solar_radiation_downwards'
]

dict_loc = {
    'Pituffik': (76.4, -68.575),
    'Fairbanks': (64.75, -147.4),
    'Guam': (13.475, 144.75),
    'Yuma_PG': (33.125, -114.125),
    'Fort_Bragg': (35.05, -79.115),
}

ERA5_1h = xr.open_zarr(
    "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3",
    chunks=None,
    storage_options=dict(token='anon'),)[varnames]

for year in range(1957, 2026):

    ERA5_select = ERA5_1h.sel(time=slice(f'{year}-01-01T00', f'{year}-12-31T23'))
    
    ERA5_select = ERA5_select.assign_coords(longitude = (((ERA5_select.longitude + 180) % 360) - 180))
    ERA5_select = ERA5_select.sortby("longitude")
    
    save_name = '/glade/campaign/ral/hap/ksha/EPRI_data/ERA5_hourly/{}_{}.zarr'
    
    for stn, (lat_mid, lon_mid) in dict_loc.items():
        
        subset = ERA5_select.sel(latitude=lat_mid, longitude=lon_mid, method='nearest')
        
        save_name_ = save_name.format(stn, year)
        subset.to_zarr(save_name_, mode='w')
        print(f'Save to {save_name_}')


# ============================================================================= #
# Convert hourly ERA5 to daily values
# ============================================================================= #

station_names = ['Pituffik', 'Fairbanks', 'Guam', 'Yuma_PG' ,'Fort_Bragg']

varname_pick = [
    '2m_temperature',
    'maximum_2m_temperature_since_previous_post_processing',
    'minimum_2m_temperature_since_previous_post_processing',
    'total_precipitation'
]

varname_rename = {
    '2m_temperature': 'TREFHT',
    'maximum_2m_temperature_since_previous_post_processing': 'TREFHTMX',
    'minimum_2m_temperature_since_previous_post_processing': 'TREFHTMN',
    'total_precipitation': 'PRECT'        
}

for station in station_names:
    
    fn_all = []
    for year in range(1957, 2026):
        fn_all.append(f'/glade/campaign/ral/hap/ksha/EPRI_data/ERA5_hourly/{station}_{year}.zarr')
    
    ds_collection = []
    for fn in fn_all:
        ds = xr.open_zarr(fn)[varname_pick]
        ds = ds.rename(varname_rename)
        ds_collection.append(ds)
        
    for i_year, year in enumerate(range(1957, 2026)):

        if i_year > 0:
            ds = ds_collection[i_year]
            # ======================================================== #
            # t2
            ds_t2 = xr.Dataset()
            ds_t2['TREFHTMX'] = ds['TREFHTMX'].resample(time="1D").max(keep_attrs=True)
            ds_t2['TREFHTMN'] = ds['TREFHTMN'].resample(time="1D").min(keep_attrs=True)
            ds_t2['TREFHT'] = ds['TREFHT'].resample(time="1D").mean(keep_attrs=True)
            
            # ======================================================== #
            # precip
            ds_previous = ds_collection[i_year-1].isel(time=slice(-48, None))
            ds_precip = xr.concat([ds_previous, ds], dim='time')

            time_start = '{}-12-31T00'.format(year-1)
            time_start_save = '{}-01-01T00'.format(year)
            time_end = '{}-12-31T23'.format(year)
            
            ds_hourly = ds_precip.sel(time=slice(time_start, time_end))
            ds_hourly = ds_hourly[['PRECT']]
            ds_hourly_shifted = ds_hourly.shift(time=-1)
            ds_daily = ds_hourly_shifted.resample(time='24h').sum()
            ds_daily['time'] = ds_daily['time'] + pd.Timedelta(hours=24)
            ds_daily = ds_daily.sel(time=slice(time_start_save, time_end))

            # ======================================================== #
            # combine & save
            ds_daily = xr.merge([ds_daily, ds_t2])
            
            save_name = f'/glade/campaign/ral/hap/ksha/EPRI_data/ERA5_daily/{station}_{year}.zarr'
            ds_daily.to_zarr(save_name, mode='w', consolidated=True, compute=True)
            print(save_name)


# ============================================================================= #
# Convert daily ERA5 to annual metrics
# ============================================================================= #

def annual_metrics(ds_in, suffix):
    """
    Compute yearly min/max/mean and 30-day rolling-mean max, then rename + suffix.
    Minimal change: same variables as your original.
    """
    g = ds_in.groupby("time.year")
    ds_max  = g.max("time", skipna=True)
    ds_min  = g.min("time", skipna=True)
    ds_mean = g.mean("time", skipna=True)
    ds_30d = (
        ds_in.rolling(time=30, min_periods=30).mean()
            .groupby("time.year").max("time", skipna=True)
    )

    ds_min = ds_min.rename({'TREFHTMN': 'TREFHTMN_min', 'TREFHT': 'TREFHT_min'})[['TREFHTMN_min', 'TREFHT_min']]
    ds_max = ds_max.rename({'PRECT': 'PRECT_max', 'TREFHTMX': 'TREFHTMX_max', 'TREFHT': 'TREFHT_max'})[['PRECT_max', 'TREFHTMX_max', 'TREFHT_max']]
    ds_30d = ds_30d.rename({'TREFHT': 'TREFHT_30d', 'PRECT': 'PRECT_30d'})[['TREFHT_30d', 'PRECT_30d']]
    ds_mean = ds_mean.rename({'PRECT': 'PRECT_mean', 'TREFHT': 'TREFHT_mean'})[['PRECT_mean', 'TREFHT_mean']]

    ds_out = xr.merge([ds_min, ds_max, ds_30d, ds_mean])
    return ds_out.rename({v: f"{v}_{suffix}" for v in ds_out.data_vars})

for stn in station_names:
    base_dir = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS_STN/{stn}/'
    
    # get data
    list_ds = []
    for year in range(1958, 2025):
        fn = f'/glade/campaign/ral/hap/ksha/EPRI_data/ERA5_daily/{stn}_{year}.zarr'
        ds = xr.open_zarr(fn)
        list_ds.append(ds)
        
    ds_all = xr.concat(list_ds, dim='time')
    ds_all = ds_all[['PRECT', 'TREFHT', 'TREFHTMX', 'TREFHTMN']]
    ds_all['PRECT'] = ds_all['PRECT'] * 1000  # mm per day
    ds_all = ds_all.chunk({"time": -1})

    # ======================= #
    # metrics (compute via helper; avoids repeating logic)
    ds_metrics_default = annual_metrics(ds_all, "default")

    # ========================== #
    # save
    ds_final = ds_metrics_default
    
    save_name = base_dir + 'metrics.zarr'
    ds_final.to_zarr(save_name, mode='w')
    print(save_name)

# ============================================================================= #
# Save CESM metrics as netCDF
# ============================================================================= #

def detrend_linear(da, dim="time"):
    """
    Remove a best-fit linear trend along `dim` for each grid point.
    Uses an index-based time axis (0..N-1) to avoid datetime scaling issues.
    """
    t = xr.DataArray(np.arange(da.sizes[dim]), dims=dim, coords={dim: da[dim]})

    valid = np.isfinite(da)
    t_valid = t.where(valid)
    da_valid = da.where(valid)

    t_mean = t_valid.mean(dim, skipna=True)
    y_mean = da_valid.mean(dim, skipna=True)

    cov = ((t_valid - t_mean) * (da_valid - y_mean)).mean(dim, skipna=True)
    var = ((t_valid - t_mean) ** 2).mean(dim, skipna=True)

    slope = cov / var
    intercept = y_mean - slope * t_mean

    trend = slope * t + intercept
    return da - trend
    

dict_loc = {
    'Pituffik': (76.4, -68.575),
    'Fairbanks': (64.75, -147.4),
    'Guam': (13.475, 144.75),
    'Yuma_PG': (33.125, -114.125),
    'Fort_Bragg': (35.05, -79.115),
}
keys = list(dict_loc.keys())

base_dir = '/glade/derecho/scratch/ksha/EPRI_data/METRICS_STN/'

lat = [coords[0] for coords in dict_loc.values()]
lon = [coords[1] for coords in dict_loc.values()]

list_ds = []
for stn in keys:
    fn_CESM = base_dir + f'{stn}/CESM_metrics.zarr'
    ds_CESM = xr.open_zarr(fn_CESM)
    ds_CESM = ds_CESM.rename({'init_time': 'initialization', 'lead_year': 'lead_time'})
    ds_CESM = ds_CESM.drop_vars(['lat', 'lon'])
    list_ds.append(ds_CESM)
    
ds_CESM_all = xr.concat(list_ds, dim='site')
ds_CESM_all = ds_CESM_all.assign_coords({'site': keys})
ds_CESM_all = ds_CESM_all.assign_coords(lat=('lat', lat))
ds_CESM_all = ds_CESM_all.assign_coords(lon=('lon', lon))


# ================================================ #
# anomaly
ds_CESM_all_anom = ds_CESM_all.copy()
vars_ = list(ds_CESM_all.keys())

for v in vars_:
    clim = ds_CESM_all[v].mean("initialization")
    ds_CESM_all_anom[v] = ds_CESM_all[v] - clim
ds_CESM_all_anom = ds_CESM_all_anom[vars_]
ds_CESM_all_anom = ds_CESM_all_anom.rename({v: v[:-7]+'anom' for v in ds_CESM_all_anom.data_vars})

# ================================================ #
# detrend
ds_CESM_all_detrend = ds_CESM_all.copy()
for v in vars_:
    ds_CESM_all_detrend[v] = detrend_linear(ds_CESM_all[v], dim="initialization")
ds_CESM_all_detrend = ds_CESM_all_detrend[vars_]
ds_CESM_all_detrend = ds_CESM_all_detrend.rename({v: v[:-7]+'detrend' for v in ds_CESM_all_detrend.data_vars})

ds_merge = xr.merge([ds_CESM_all, ds_CESM_all_anom, ds_CESM_all_detrend])
ds_merge = ds_merge.rename({'initialization': 'gen_date'})
ds_merge = ds_merge.load()

# ================================================ #
# NINO 3.4
fn = '/glade/campaign/ral/hap/ksha/EPRI_data/CESM_OCN/ENSO_index.npy'
NINO34 = np.load(fn, allow_pickle=True)[()]

years = np.array(list(NINO34.keys()))

nino_index = np.empty((5, 10, 62))

for i_year, year in enumerate(years):
    nino_index[:, :, i_year] = NINO34[int(year)][2:][None, :]

ds_merge["nino_34"] = xr.DataArray(
    nino_index,
    dims=("site", "lead_time", "gen_date"),
    coords={
        "site": ds_merge .site,
        "lead_time": ds_merge .lead_time,
        "gen_date": ds_merge .gen_date,
    },)

# ================================================ #
# NAO
ds_NAO = xr.open_zarr('/glade/derecho/scratch/ksha/EPRI_data/METRICS/NAO.zarr')
ds_NAO = ds_NAO.load()
ds_NAO = ds_NAO.rename({'lead_year': 'lead_time', 'init_year': 'gen_date'})
ds_NAO = ds_NAO.rename({'nao_ann_eof': 'NAO_annual_mean', 'nao_djf_eof': 'NAO_DJF'})[['NAO_annual_mean', 'NAO_DJF']]
ds_NAO = ds_NAO.expand_dims({'site': 5})
ds_NAO = ds_NAO.drop_vars(('mode',))
ds_NAO = ds_NAO.assign_coords({'gen_date': ds_merge['gen_date']})
ds_merge = xr.merge([ds_merge, ds_NAO])

# ================================================ #
# GBI
ds_GBI = xr.open_zarr('/glade/derecho/scratch/ksha/EPRI_data/METRICS/GBI.zarr')
ds_GBI = ds_GBI.load()
ds_GBI = ds_GBI.rename({'init_year': 'gen_date', 'lead_year': 'lead_time'})
ds_GBI = ds_GBI.expand_dims({'site': 5})
ds_GBI = ds_GBI.assign_coords({'gen_date': ds_merge['gen_date']})
ds_merge = xr.merge([ds_merge, ds_GBI])

ds_merge['GBI_30d'] = ds_merge['GBI_30d'].transpose('site', 'lead_time', 'gen_date')
ds_merge['GBI_mean'] = ds_merge['GBI_mean'].transpose('site', 'lead_time', 'gen_date')

rename_dict = {
    'PRECT_30d_default'   : 'CESM_precip_annual_max_30d',
    'PRECT_max_default'   : 'CESM_precip_max_daily',
    'PRECT_mean_default'  : 'CESM_precip_mean',
    'TREFHTMN_min_default': 'CESM_t2m_min_hour',
    'TREFHTMX_max_default': 'CESM_t2m_max_hour',
    'TREFHT_30d_default'  : 'CESM_t2m_annual_max_30d',
    'TREFHT_max_default'  : 'CESM_t2m_max_daily',
    'TREFHT_mean_default' : 'CESM_t2m_mean',
    'TREFHT_min_default'  : 'CESM_t2m_min_daily',
    'PRECT_30d_anom'      : 'CESM_anomaly_precip_annual_max_30d',
    'PRECT_max_anom'      : 'CESM_anomaly_precip_max_daily',
    'PRECT_mean_anom'     : 'CESM_anomaly_precip_mean',
    'TREFHTMN_min_anom'   : 'CESM_anomaly_t2m_min_hour',
    'TREFHTMX_max_anom'   : 'CESM_anomaly_t2m_max_hour',
    'TREFHT_30d_anom'     : 'CESM_anomaly_t2m_annual_max_30d',
    'TREFHT_max_anom'     : 'CESM_anomaly_t2m_max_daily',
    'TREFHT_mean_anom'    : 'CESM_anomaly_t2m_mean',
    'TREFHT_min_anom'     : 'CESM_anomaly_t2m_min_daily',
    'PRECT_30d_detrend'   : 'CESM_detrend_precip_annual_max_30d',
    'PRECT_max_detrend'   : 'CESM_detrend_precip_max_daily',
    'PRECT_mean_detrend'  : 'CESM_detrend_precip_mean',
    'TREFHTMN_min_detrend': 'CESM_detrend_t2m_min_hour',
    'TREFHTMX_max_detrend': 'CESM_detrend_t2m_max_hour',
    'TREFHT_30d_detrend'  : 'CESM_detrend_t2m_annual_max_30d',
    'TREFHT_max_detrend'  : 'CESM_detrend_t2m_max_daily',
    'TREFHT_mean_detrend' : 'CESM_detrend_t2m_mean',
    'TREFHT_min_detrend'  : 'CESM_detrend_t2m_min_daily',
    'nino_34'             : 'nino_34'
}

ds_merge = ds_merge.rename(rename_dict)
ds_merge.to_netcdf(
    '/glade/derecho/scratch/ksha/EPRI_AnEn/input_AnEn_CESM_20260312.nc',
    format="NETCDF4_CLASSIC",
    engine="netcdf4",
    mode='w'
)

# ============================================================================= #
# Save ERA5 metrics as netCDF
# ============================================================================= #

list_ds = []
for stn in keys:

    fn_ERA5 = base_dir + f'{stn}/metrics.zarr'
    ds_ERA5 = xr.open_zarr(fn_ERA5)
    ds_ERA5 = ds_ERA5.drop_vars(['latitude', 'longitude'])
    years_ext = np.arange(int(ds_ERA5["year"].min()), 2036)  # 1958..2035 inclusive
    ds_ERA5 = ds_ERA5.reindex(year=years_ext)
    
    fn_CESM = base_dir + f'{stn}/CESM_metrics.zarr'
    ds_CESM = xr.open_zarr(fn_CESM)
    ds_CESM = ds_CESM.drop_vars(['lat', 'lon'])
    
    valid_year = (ds_CESM["init_time"] + ds_CESM["lead_year"]).rename("valid_year") 
    valid_year = valid_year.transpose("lead_year", "init_time") 
    ds_target = ds_ERA5.sel(year=valid_year) 
    ds_target = ds_target.drop_vars("year") #.assign_coords(valid_year=valid_year)
    list_ds.append(ds_target)
    
ds_target_all = xr.concat(list_ds, dim='site')
ds_target_all = ds_target_all.assign_coords({'site': keys})
ds_target_all = ds_target_all.assign_coords(lat=('site', lat))
ds_target_all = ds_target_all.assign_coords(lon=('site', lon))
ds_target_all = ds_target_all.rename({'lead_year': 'lead_time', 'init_time': 'gen_date'})

# ================================================ #
# anomaly
ds_target_all_anom = ds_target_all.copy()
vars_ = list(ds_target_all.keys())

for v in vars_:
    clim = ds_target_all[v].mean('gen_date', skipna=True)
    ds_target_all_anom[v] = ds_target_all[v] - clim
ds_target_all_anom = ds_target_all_anom[vars_]
ds_target_all_anom = ds_target_all_anom.rename({v: v[:-7]+'anom' for v in ds_target_all_anom.data_vars})

# ================================================ #
# detrend
ds_target_all_detrend = ds_target_all.copy()
for v in vars_:
    ds_target_all_detrend[v] = detrend_linear(ds_target_all[v], dim='gen_date')
ds_target_all_detrend = ds_target_all_detrend[vars_]
ds_target_all_detrend = ds_target_all_detrend.rename({v: v[:-7]+'detrend' for v in ds_target_all_detrend.data_vars})

ds_target_merge = xr.merge([ds_target_all, ds_target_all_anom, ds_target_all_detrend])

rename_dict = {
    'PRECT_30d_default'   : 'ERA5_precip_annual_max_30d',
    'PRECT_max_default'   : 'ERA5_precip_max_daily',
    'PRECT_mean_default'  : 'ERA5_precip_mean',
    'TREFHTMN_min_default': 'ERA5_t2m_min_hour',
    'TREFHTMX_max_default': 'ERA5_t2m_max_hour',
    'TREFHT_30d_default'  : 'ERA5_t2m_annual_max_30d',
    'TREFHT_max_default'  : 'ERA5_t2m_max_daily',
    'TREFHT_mean_default' : 'ERA5_t2m_mean',
    'TREFHT_min_default'  : 'ERA5_t2m_min_daily',
    'PRECT_30d_anom'      : 'ERA5_anomaly_precip_annual_max_30d',
    'PRECT_max_anom'      : 'ERA5_anomaly_precip_max_daily',
    'PRECT_mean_anom'     : 'ERA5_anomaly_precip_mean',
    'TREFHTMN_min_anom'   : 'ERA5_anomaly_t2m_min_hour',
    'TREFHTMX_max_anom'   : 'ERA5_anomaly_t2m_max_hour',
    'TREFHT_30d_anom'     : 'ERA5_anomaly_t2m_annual_max_30d',
    'TREFHT_max_anom'     : 'ERA5_anomaly_t2m_max_daily',
    'TREFHT_mean_anom'    : 'ERA5_anomaly_t2m_mean',
    'TREFHT_min_anom'     : 'ERA5_anomaly_t2m_min_daily',
    'PRECT_30d_detrend'   : 'ERA5_detrend_precip_annual_max_30d',
    'PRECT_max_detrend'   : 'ERA5_detrend_precip_max_daily',
    'PRECT_mean_detrend'  : 'ERA5_detrend_precip_mean',
    'TREFHTMN_min_detrend': 'ERA5_detrend_t2m_min_hour',
    'TREFHTMX_max_detrend': 'ERA5_detrend_t2m_max_hour',
    'TREFHT_30d_detrend'  : 'ERA5_detrend_t2m_annual_max_30d',
    'TREFHT_max_detrend'  : 'ERA5_detrend_t2m_max_daily',
    'TREFHT_mean_detrend' : 'ERA5_detrend_t2m_mean',
    'TREFHT_min_detrend'  : 'ERA5_detrend_t2m_min_daily',
}

ds_target_merge = ds_target_merge.rename(rename_dict)

# NaN to -9999
fill = -9999.0

encoding = {}
for v in ds_target_merge.data_vars:
    encoding[v] = {"_FillValue": fill}

ds_target_merge.to_netcdf(
    '/glade/derecho/scratch/ksha/EPRI_AnEn/input_AnEn_ERA5_20260226.nc',
    format="NETCDF4_CLASSIC",
    engine="netcdf4",
    encoding=encoding,
    mode='w'
)

